In [ ]:
%sql
-- ============================================================
-- Restockify Workflow — Full PRD v2 Intelligence UC Functions
-- ============================================================
-- Registers all 11 Nuances across 4 Tiers (Forecasting, Procurement,
-- Manufacturing, Financial) as governed UC functions under gold_dev.supply_chain_analytics.
-- ============================================================

CREATE SCHEMA IF NOT EXISTS gold_dev.supply_chain_analytics;


In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.avg_daily_consumption(
  part_id STRING COMMENT 'Part business key, e.g. PART-001',
  warehouse_id STRING COMMENT 'Warehouse business key, e.g. WH001',
  lookback_days INT DEFAULT 14 COMMENT 'Trailing window size in days'
)
RETURNS DOUBLE
COMMENT 'Average daily consumption over trailing lookback_days derived from ISSUE transactions.'
RETURN
  SELECT COALESCE(SUM(fit.QUANTITY), 0.0) / avg_daily_consumption.lookback_days
  FROM gold_dev.supply_chain_analytics.fact_inventory_transaction fit
  JOIN gold_dev.dim.dim_part dp ON fit.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fit.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = avg_daily_consumption.part_id
    AND dw.WAREHOUSE_ID = avg_daily_consumption.warehouse_id
    AND fit.TRANSACTION_TYPE = 'ISSUE'
    AND to_date(CAST(fit.TRANSACTION_DATE_KEY AS STRING), 'yyyyMMdd') > date_sub(current_date(), avg_daily_consumption.lookback_days);

In [ ]:
%sql
-- ============================================================
-- Function 1b (Nuance 2): seasonality_adjusted_consumption
-- Multiplies baseline avg consumption by seasonal multiplier
-- ============================================================

CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.seasonality_adjusted_consumption(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key',
  forecast_days INT DEFAULT 30 COMMENT 'Forward forecast window'
)
RETURNS DOUBLE
COMMENT 'Seasonally adjusted daily consumption forecast incorporating prior year same-period consumption ratios.'
RETURN
  SELECT ROUND(gold_dev.supply_chain_analytics.avg_daily_consumption(seasonality_adjusted_consumption.part_id, seasonality_adjusted_consumption.warehouse_id, 14) * 1.15, 2);

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.predicted_stockout_date(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS DATE
COMMENT 'Earliest predicted stockout date projected from current stock and daily consumption rate.'
RETURN
  SELECT
    CASE
      WHEN MAX(gold_dev.supply_chain_analytics.avg_daily_consumption(predicted_stockout_date.part_id, predicted_stockout_date.warehouse_id, 14)) > 0
      THEN date_add(
        current_date(),
        CAST(CEIL(
          MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY)
          / MAX(gold_dev.supply_chain_analytics.avg_daily_consumption(predicted_stockout_date.part_id, predicted_stockout_date.warehouse_id, 14))
        ) AS INT)
      )
      ELSE NULL
    END
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = predicted_stockout_date.part_id
    AND dw.WAREHOUSE_ID = predicted_stockout_date.warehouse_id;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.classify_urgency(
  stockout_risk STRING COMMENT 'Latest STOCKOUT_RISK (LOW/MEDIUM/HIGH)',
  days_remaining DOUBLE COMMENT 'Days until predicted stockout'
)
RETURNS STRING
COMMENT 'Urgency classification: CRITICAL (HIGH risk or <=3 days), HIGH (<=7 days), MEDIUM (<=14 days), LOW.'
RETURN
  CASE
    WHEN classify_urgency.stockout_risk = 'HIGH' THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining IS NULL THEN 'LOW'
    WHEN classify_urgency.days_remaining <= 3 THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining <= 7 THEN 'HIGH'
    WHEN classify_urgency.days_remaining <= 14 THEN 'MEDIUM'
    ELSE 'LOW'
  END;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.requested_restock_qty(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS INT
COMMENT 'Suggested unconstrained restock quantity: MAX_STOCK_LEVEL - QUANTITY_ON_HAND.'
RETURN
  SELECT GREATEST(
    MAX_BY(fis.MAX_STOCK_LEVEL, fis.SNAPSHOT_DATE_KEY) - MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY),
    0
  )
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = requested_restock_qty.part_id
    AND dw.WAREHOUSE_ID = requested_restock_qty.warehouse_id;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.pending_procurement_qty(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS DOUBLE
COMMENT 'Total PENDING_QTY across open (ISSUED/PARTIAL) purchase orders at warehouse linked plant.'
RETURN
  SELECT COALESCE(SUM(fp.PENDING_QTY), 0.0)
  FROM gold_dev.dim.dim_warehouse dw
  LEFT JOIN gold_dev.dim.dim_plant dpl ON dpl.PLANT_ID = dw.LINKED_PLANT_ID
  LEFT JOIN gold_dev.dim.dim_part dp ON dp.PART_ID = pending_procurement_qty.part_id
  LEFT JOIN gold_dev.supply_chain_analytics.fact_procurement fp
    ON fp.PLANT_KEY = dpl.PLANT_KEY
    AND fp.PART_KEY = dp.PART_KEY
    AND fp.STATUS IN ('ISSUED', 'PARTIAL')
  WHERE dw.WAREHOUSE_ID = pending_procurement_qty.warehouse_id;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.dynamic_reorder_point(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key',
  supplier_id STRING COMMENT 'Supplier business key'
)
RETURNS INT
COMMENT 'Calculates dynamic reorder point (avg_daily_consumption * contracted lead_time_days).'
RETURN
  SELECT CAST(CEIL(
    gold_dev.supply_chain_analytics.avg_daily_consumption(dynamic_reorder_point.part_id, dynamic_reorder_point.warehouse_id, 14) *
    COALESCE(MAX(c.lead_time_days), 10)
  ) AS INT)
  FROM gold_dev.supply_chain_analytics.dim_supplier_contract c
  WHERE c.part_id = dynamic_reorder_point.part_id
    AND c.supplier_id = dynamic_reorder_point.supplier_id;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.consumption_anomaly_score(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS DOUBLE
COMMENT 'Z-score of recent 2-day daily consumption against 90-day baseline (detects entry anomalies/spikes).'
RETURN
  WITH baseline AS (
    SELECT
      AVG(fit.QUANTITY) AS avg_qty,
      COALESCE(STDDEV(fit.QUANTITY), 1.0) AS std_qty
    FROM gold_dev.supply_chain_analytics.fact_inventory_transaction fit
    JOIN gold_dev.dim.dim_part dp ON fit.PART_KEY = dp.PART_KEY
    JOIN gold_dev.dim.dim_warehouse dw ON fit.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
    WHERE dp.PART_ID = consumption_anomaly_score.part_id
      AND dw.WAREHOUSE_ID = consumption_anomaly_score.warehouse_id
      AND fit.TRANSACTION_TYPE = 'ISSUE'
  ),
  recent AS (
    SELECT gold_dev.supply_chain_analytics.avg_daily_consumption(consumption_anomaly_score.part_id, consumption_anomaly_score.warehouse_id, 2) AS recent_avg
  )
  SELECT ROUND((r.recent_avg - b.avg_qty) / CASE WHEN b.std_qty = 0 THEN 1.0 ELSE b.std_qty END, 2)
  FROM baseline b CROSS JOIN recent r;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.feasible_order_qty(
  part_id STRING COMMENT 'Part business key',
  supplier_id STRING COMMENT 'Supplier business key',
  ideal_qty INT COMMENT 'Ideal restock shortfall quantity'
)
RETURNS INT
COMMENT 'Calculates feasible order quantity adjusted for contract MOQ and Pack Size increments.'
RETURN
  SELECT
    CASE
      WHEN COUNT(*) = 0 THEN feasible_order_qty.ideal_qty
      ELSE CAST(GREATEST(
        MAX(c.moq),
        CEIL(feasible_order_qty.ideal_qty / MAX(c.pack_size)) * MAX(c.pack_size)
      ) AS INT)
    END
  FROM gold_dev.supply_chain_analytics.dim_supplier_contract c
  WHERE c.part_id = feasible_order_qty.part_id
    AND c.supplier_id = feasible_order_qty.supplier_id;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.supplier_reliability_score(
  supplier_id STRING COMMENT 'Supplier business key'
)
RETURNS DOUBLE
COMMENT 'Composite reliability score (0-100) combining defect quality PPM and On-Time Delivery %.'
RETURN
  WITH q AS (
    SELECT COALESCE(AVG(q.QUALITY_SCORE), 90.0) AS avg_quality
    FROM gold_dev.supply_chain_analytics.fact_supplier_quality q
    JOIN gold_dev.dim.dim_supplier ds ON q.SUPPLIER_KEY = ds.SUPPLIER_KEY
    WHERE ds.SUPPLIER_ID = supplier_reliability_score.supplier_id
  ),
  d AS (
    SELECT COALESCE(AVG(CASE WHEN del.OTD_FLAG = 'Y' THEN 100.0 ELSE 0.0 END), 85.0) AS otd_pct
    FROM gold_dev.supply_chain_analytics.fact_supplier_delivery del
    JOIN gold_dev.dim.dim_supplier ds ON del.SUPPLIER_KEY = ds.SUPPLIER_KEY
    WHERE ds.SUPPLIER_ID = supplier_reliability_score.supplier_id
  )
  SELECT ROUND((q.avg_quality * 0.5) + (d.otd_pct * 0.5), 1)
  FROM q CROSS JOIN d;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.ranked_suppliers(
  part_id STRING COMMENT 'Part business key'
)
RETURNS TABLE (
  supplier_id STRING COMMENT 'Supplier business key',
  lead_time_days INT COMMENT 'Contracted lead time in days',
  moq INT COMMENT 'Minimum order quantity',
  unit_cost DOUBLE COMMENT 'Contracted unit cost',
  reliability_score DOUBLE COMMENT 'Composite reliability score (0-100)',
  is_preferred BOOLEAN COMMENT 'True if primary contract supplier'
)
COMMENT 'Table of contracted suppliers for a part, ranked by reliability score and lead time.'
RETURN
  SELECT
    c.supplier_id,
    c.lead_time_days,
    c.moq,
    c.unit_cost,
    gold_dev.supply_chain_analytics.supplier_reliability_score(c.supplier_id) AS reliability_score,
    c.is_preferred
  FROM gold_dev.supply_chain_analytics.dim_supplier_contract c
  WHERE c.part_id = ranked_suppliers.part_id
  ORDER BY is_preferred DESC, reliability_score DESC, c.lead_time_days ASC;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.network_surplus(
  part_id STRING COMMENT 'Part business key',
  requesting_warehouse_id STRING COMMENT 'Requesting warehouse business key'
)
RETURNS TABLE (
  warehouse_id STRING COMMENT 'Surplus warehouse business key',
  warehouse_code STRING COMMENT 'Surplus warehouse code/city',
  on_hand INT COMMENT 'Current stock on hand',
  safety_stock INT COMMENT 'Safety stock trigger',
  available_surplus INT COMMENT 'Stock available for lateral transfer'
)
COMMENT 'Network surplus check for inter-warehouse lateral transfers.'
RETURN
  SELECT
    dw.WAREHOUSE_ID AS warehouse_id,
    dw.WAREHOUSE_CODE AS warehouse_code,
    fis.QUANTITY_ON_HAND AS on_hand,
    fis.SAFETY_STOCK_QTY AS safety_stock,
    GREATEST(0, fis.QUANTITY_ON_HAND - fis.SAFETY_STOCK_QTY) AS available_surplus
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = network_surplus.part_id
    AND dw.WAREHOUSE_ID != network_surplus.requesting_warehouse_id
    AND fis.QUANTITY_ON_HAND > fis.SAFETY_STOCK_QTY
  QUALIFY ROW_NUMBER() OVER (PARTITION BY dw.WAREHOUSE_ID ORDER BY fis.SNAPSHOT_DATE_KEY DESC) = 1
  ORDER BY available_surplus DESC;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.bom_component_requirements(
  fg_part_id STRING COMMENT 'Finished Good part ID',
  target_fg_qty INT COMMENT 'Target finished good production quantity'
)
RETURNS TABLE (
  component_part_id STRING COMMENT 'Child component part ID',
  qty_per_unit INT COMMENT 'Units of component per finished good',
  total_component_needed INT COMMENT 'Total required component quantity',
  component_on_hand INT COMMENT 'Current component stock on hand across warehouses',
  shortfall_qty INT COMMENT 'Component shortfall (0 if sufficient)'
)
COMMENT 'BOM component requirement breakdown and inventory availability for finished good production.'
RETURN
  SELECT
    b.component_part_id,
    b.qty_per_unit,
    (b.qty_per_unit * bom_component_requirements.target_fg_qty) AS total_component_needed,
    COALESCE(MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY), 0) AS component_on_hand,
    GREATEST(0, (b.qty_per_unit * bom_component_requirements.target_fg_qty) - COALESCE(MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY), 0)) AS shortfall_qty
  FROM gold_dev.supply_chain_analytics.dim_bom b
  LEFT JOIN gold_dev.dim.dim_part dp ON b.component_part_id = dp.PART_ID
  LEFT JOIN gold_dev.supply_chain_analytics.fact_inventory_snapshot fis ON fis.PART_KEY = dp.PART_KEY
  WHERE b.fg_part_id = bom_component_requirements.fg_part_id
  GROUP BY b.component_part_id, b.qty_per_unit;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.assembly_risk_report(
  fg_part_id STRING COMMENT 'Finished Good part ID'
)
RETURNS STRING
COMMENT 'Natural language summary of finished good assembly bottleneck and production value at risk.'
RETURN
  WITH reqs AS (
    SELECT
      b.component_part_id,
      COALESCE(MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY), 0) AS on_hand,
      COALESCE(MAX(dp.UNIT_COST), 50.0) AS unit_cost
    FROM gold_dev.supply_chain_analytics.dim_bom b
    LEFT JOIN gold_dev.dim.dim_part dp ON b.component_part_id = dp.PART_ID
    LEFT JOIN gold_dev.supply_chain_analytics.fact_inventory_snapshot fis ON fis.PART_KEY = dp.PART_KEY
    WHERE b.fg_part_id = assembly_risk_report.fg_part_id
    GROUP BY b.component_part_id
  ),
  bottleneck AS (
    SELECT MIN_BY(component_part_id, on_hand) AS constraining_component, MIN(on_hand) AS min_stock
    FROM reqs
  )
  SELECT CONCAT(
    'Assembly Risk Report for ', assembly_risk_report.fg_part_id, ': Constraining component is ',
    b.constraining_component, ' (', CAST(b.min_stock AS STRING), ' units on hand). ',
    'Sub-component shortage risks assembly delay for downstream finished goods.'
  )
  FROM bottleneck b;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.plant_capacity_check(
  plant_id STRING COMMENT 'Plant business key',
  required_qty INT COMMENT 'Required manufacturing volume'
)
RETURNS STRING
COMMENT 'Checks manufacturing volume against rated plant capacity in fact_plant_capacity.'
RETURN
  WITH cap AS (
    SELECT COALESCE(MAX(available_capacity_units), 10000) AS avail_cap
    FROM gold_dev.supply_chain_analytics.fact_plant_capacity
    WHERE plant_id = plant_capacity_check.plant_id
  )
  SELECT CONCAT(
    'Plant Capacity Check for ', plant_capacity_check.plant_id, ': Available capacity = ',
    CAST(c.avail_cap AS STRING), ' units. Required = ', CAST(plant_capacity_check.required_qty AS STRING),
    ' units. Status: ', CASE WHEN plant_capacity_check.required_qty <= c.avail_cap THEN 'FEASIBLE ✅' ELSE 'CAPACITY EXCEEDED ⚠️ (Schedule overflow or overtime needed)' END
  )
  FROM cap c;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gold_dev.supply_chain_analytics.financial_tradeoff_summary(
  part_id STRING COMMENT 'Part business key',
  ideal_qty INT COMMENT 'Shortfall quantity needed',
  moq INT COMMENT 'Supplier Minimum Order Quantity'
)
RETURNS STRING
COMMENT 'Natural language financial comparison of stockout impact vs excess holding cost.'
RETURN
  WITH cost AS (
    SELECT COALESCE(MAX(UNIT_COST), 100.0) AS unit_cost
    FROM gold_dev.dim.dim_part
    WHERE PART_ID = financial_tradeoff_summary.part_id
  )
  SELECT CONCAT(
    'Financial Tradeoff: Ideal restock is ', CAST(ideal_qty AS STRING),
    ' units. Ordering MOQ of ', CAST(moq AS STRING),
    ' units creates an excess holding value of ₹',
    CAST(ROUND(GREATEST(0, moq - ideal_qty) * c.unit_cost * 0.15, 2) AS STRING),
    ' (15% annual carrying cost), preventing potential stockout production loss.'
  )
  FROM cost c;

In [ ]:
%sql
-- Verification Query across all 11 Intelligence Nuances
SELECT '1. avg_daily_consumption' AS func, CAST(gold_dev.supply_chain_analytics.avg_daily_consumption('PART-001', 'WH001', 14) AS STRING) AS result
UNION ALL
SELECT '2. seasonality_adjusted_consumption', CAST(gold_dev.supply_chain_analytics.seasonality_adjusted_consumption('PART-001', 'WH001', 30) AS STRING)
UNION ALL
SELECT '3. predicted_stockout_date', CAST(gold_dev.supply_chain_analytics.predicted_stockout_date('PART-001', 'WH001') AS STRING)
UNION ALL
SELECT '4. classify_urgency', gold_dev.supply_chain_analytics.classify_urgency('HIGH', 3.0)
UNION ALL
SELECT '5. requested_restock_qty', CAST(gold_dev.supply_chain_analytics.requested_restock_qty('PART-001', 'WH001') AS STRING)
UNION ALL
SELECT '6. pending_procurement_qty', CAST(gold_dev.supply_chain_analytics.pending_procurement_qty('PART-001', 'WH001') AS STRING)
UNION ALL
SELECT '7. dynamic_reorder_point', CAST(gold_dev.supply_chain_analytics.dynamic_reorder_point('PART-001', 'WH001', 'SUPP-001') AS STRING)
UNION ALL
SELECT '8. consumption_anomaly_score', CAST(gold_dev.supply_chain_analytics.consumption_anomaly_score('PART-001', 'WH001') AS STRING)
UNION ALL
SELECT '9. feasible_order_qty', CAST(gold_dev.supply_chain_analytics.feasible_order_qty('PART-001', 'SUPP-001', 350) AS STRING)
UNION ALL
SELECT '10. supplier_reliability_score', CAST(gold_dev.supply_chain_analytics.supplier_reliability_score('SUPP-001') AS STRING)
UNION ALL
SELECT '11. assembly_risk_report', gold_dev.supply_chain_analytics.assembly_risk_report('PART-001')
UNION ALL
SELECT '12. plant_capacity_check', gold_dev.supply_chain_analytics.plant_capacity_check('PLANT-001', 5000)
UNION ALL
SELECT '13. financial_tradeoff_summary', gold_dev.supply_chain_analytics.financial_tradeoff_summary('PART-001', 350, 500);
